# GNN Training and Serving Pipeline Test

This notebook demonstrates how to execute the GNN training pipeline and serving inference code entirely within a local notebook environment, hitting your actual Google Spanner database for data without deploying to Vertex AI or storing artifacts in Google Cloud Storage.

It uses the exact same code blocks found in:
- `gnn/src/pipeline/run_pipeline_local.py`
- `gnn/src/train_hetgnn.py`
- `gnn/src/serve.py`

In [3]:
import os
import sys
import asyncio
import argparse
from pathlib import Path
from unittest.mock import patch

# Ensure the gnn/src directory is in the Python path so we can import modules
src_path = str((Path.cwd().resolve().parent / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import train_hetgnn
import serve
import run_pipeline_local

## Configuration

Define the parameters for Spanner connectivity. Update these as needed for your specific GCP environment.

In [4]:
project_id = os.getenv("GOOGLE_CLOUD_PROJECT", "agents-1234")
spanner_instance = os.getenv("SPANNER_INSTANCE", "networktopology-instance")
spanner_database = os.getenv("SPANNER_DATABASE", "networktopology-db")
interval_minutes = 5
num_snapshots = 10  # Reduce this for faster notebook testing

args = argparse.Namespace(
    project=project_id,
    spanner_instance=spanner_instance,
    spanner_database=spanner_database,
    interval_minutes=interval_minutes,
    num_snapshots=num_snapshots
)

output_dir = Path("notebook_artifacts")
output_dir.mkdir(exist_ok=True)
snapshots_dir = output_dir / "snapshots"

## 1. Ingest Data

Fetch network topology and metrics directly from your Spanner database using the SCD Type 2 logic inside `SpannerDataset`.

In [ ]:
print("Fetching real data from Spanner...")
snapshots = run_pipeline_local.step_ingest(args, snapshots_dir)
print(f"Ingested {len(snapshots)} snapshots.")

## 2. Train HetGNN

Pass the fetched snapshots into the training script. This step:
1. Fits the `StandardScaler` on numerical features.
2. Converts data into PyTorch `HeteroData` graphs.
3. Trains the autoencoder over the specified number of epochs.
4. Saves `model.pth`, `scalers.pkl`, and `model_stats.pth` into our local `notebook_artifacts` folder.

In [ ]:
print("Training HetGNN on downloaded snapshots...")
model, val_loss, gb = train_hetgnn.train_hetgnn_on_snapshots(
    snapshot_objects=snapshots,
    output_dir=str(output_dir),
    hidden_channels=64, # Matches production defaults
    num_layers=2,
    epochs_override=10  # Reduced for rapid feedback (default is 50)
)
print(f"Training completed. Validation Loss: {val_loss:.4f}")

## 3. Serve (Inference)

Test the production serving code.
In production, `serve.py` queries Google Cloud Storage to find the latest trained model (via `latest_run.json`). To test it locally, we temporarily patch those loader functions to point to the artifacts we just trained in our `notebook_artifacts` directory.

This execution will:
1. Reload the model and scalers we just saved.
2. Fetch one *new* real snapshot from Spanner.
3. Perform a forward pass through the model to compute node embeddings and MSE reconstruction error.
4. **Write the results** directly back to the Spanner `NodeEmbedding` table.
5. Print the anomaly scores.

In [ ]:
print("Setting up serving components with local artifacts instead of GCS...")

def mock_load_manifest():
    return {"run_id": "notebook_test_run"}

def mock_download_artefacts(manifest):
    return output_dir

async def test_serving():
    # We use unittest.mock.patch so serve.py doesn't try to download from GCS
    with patch("serve._load_manifest", side_effect=mock_load_manifest), \
         patch("serve._download_artefacts", side_effect=mock_download_artefacts), \
         patch("serve.SPANNER_INSTANCE", spanner_instance), \
         patch("serve.SPANNER_DATABASE", spanner_database), \
         patch("serve.GOOGLE_CLOUD_PROJECT", project_id):
         
        # Run the actual inference code
        results = await serve._run_inference()
        
        print("\n--- INFERENCE RESULTS ---")
        print(f"Snapshot timestamp: {results['snapshot_timestamp']}")
        print(f"Total Anomalies detected: {results['anomaly_count']}")
        for ntype, scores in results['anomaly_scores'].items():
            print(f"  {ntype} nodes: {len(scores)}")

# Await the async server inference method
await test_serving()